This example is used to illustrate the **Quantum Metal --> pyPalace** workflow as well as some basic features of pyPalace. Particuarly, we focus on the built-in pyPalace meshing functionality (backend Gmsh) with a working quantum metal example. 

**Note: Quantum Metal (formly known as Qiskit Metal) is NOT a pyPalace dependency.** 
If you want to use this workflow, install Quantum metal separately. 

## Build Quantum Metal Design

Below we build a default Quantum Metal TransmonCross with an added rectangle for a Josephson junction lumped port.

In [1]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross
from qiskit_metal.toolbox_python.attr_dict import Dict
from qiskit_metal.qlibrary.core import QComponent

In [2]:
design = designs.DesignPlanar()
gui = MetalGUI(design)

''' add the qubit geometry '''
qubit_options = dict(connection_pads=dict(claw = dict(connector_location = '90', connector_type = '0')))
qubit = TransmonCross(design,'qubit',options=qubit_options)

gui.rebuild() ## need to do this first before the JJ rectangle below will render
 
''' add a rectangular, which we will later define as our Josephson junction lumped port '''
JJ_coordinates = list(design.qgeometry.tables["junction"].geometry.iloc[0].coords)
JJ_y1,JJ_y2 = JJ_coordinates[0][1],JJ_coordinates[1][1]
JJ_y = (JJ_y1 + JJ_y2)/2
JJ_length = abs(JJ_y2-JJ_y1)
JJ_width = float(qubit.options["cross_width"][:-2])*1e-3
port_geometry = draw.rectangle(JJ_width,JJ_length,0,JJ_y)

## add the port to our design ##
design.qgeometry.add_qgeometry("poly",qubit.id,{"josephson_junction": port_geometry},layer=1,subtract=False)

gui.refresh()
gui.autoscale()

### Take a look at our design's polygon qgeometry table 

we will need this to help define what polygons we want to add as attributes to our mesh file. We only want to add surfaces which we will give boundary conditions to later on using pyPalace. 

In [3]:
design.qgeometry.tables["poly"]

,component,name,geometry,layer,subtract,helper,chip,fillet
0,1,cross,"POLYGON ((-0.2 -0.01, -0.2 0.01, -0.01 0.01, -...",1,False,False,main,NaN
1,1,cross_etch,"POLYGON ((-0.22 -0.03, -0.22 0.03, -0.03 0.03,...",1,True,False,main,NaN
2,1,claw_connector_arm,"POLYGON ((0.005 0.281, 0.005 0.241, 0.051 0.24...",1,False,False,main,NaN
3,1,claw_connector_etcher,"POLYGON ((0.011 0.287, 0.011 0.247, 0.057 0.24...",1,True,False,main,NaN
4,1,josephson_junction,"POLYGON ((-0.01 -0.22, 0.01 -0.22, 0.01 -0.2, ...",1,False,False,main,NaN


## Mesh the design with pyPalace (Gmsh backend)

In [4]:
from pypalace import mesh

choose which Quantum Metal components we want to give with mesh atrributes IDs to.

For eigenmode modeling of a superconducting qubit, we need:

| Geometry | Boundary Condition |
|---|---|
| `cross` | PEC |
| `claw_connector_arm` | PEC |
| `josephson_junction` | LumpedPort |

We will also need a `far_field` and a `ground_plane` - but these are automatically generated using ```mesh.mesh_Quantum_Metal_design()```. See below.

In [5]:
''' Mesh our Quantum Metal design '''

## define which components we want to give mesh attributes ##
Attributes = {"cross":1,"claw_connector_arm":2,"josephson_junction":3}

mesh_dir = "mesh/"
meshfile = "example00.msh"
path_to_mesh = mesh_dir + meshfile

## mesh the design --> returns pandas.dataframe of labeled mesh attributes ##
## look at readthedocs, mesh_Quantum_Metal_design has a lot of flexibility to control surface mesh size, volume mesh size, substrate size, etc ## 
attributes_table = mesh.mesh_Quantum_Metal_design(design,output_mesh=path_to_mesh,Attributes=Attributes,surface_mesh_size=0.0005);

Info    : Meshing 1D...nts                                                                                                                
Info    : [  0%] Meshing curve 49 (Line)
Info    : [ 10%] Meshing curve 50 (Line)
Info    : [ 10%] Meshing curve 51 (Line)
Info    : [ 10%] Meshing curve 52 (Line)
Info    : [ 10%] Meshing curve 53 (Line)
Info    : [ 10%] Meshing curve 54 (Line)
Info    : [ 10%] Meshing curve 55 (Line)
Info    : [ 10%] Meshing curve 56 (Line)
Info    : [ 20%] Meshing curve 57 (Line)
Info    : [ 20%] Meshing curve 58 (Line)
Info    : [ 20%] Meshing curve 59 (Line)
Info    : [ 20%] Meshing curve 60 (Line)
Info    : [ 20%] Meshing curve 61 (Line)
Info    : [ 20%] Meshing curve 62 (Line)
Info    : [ 20%] Meshing curve 63 (Line)
Info    : [ 30%] Meshing curve 64 (Line)
Info    : [ 30%] Meshing curve 65 (Line)
Info    : [ 30%] Meshing curve 66 (Line)
Info    : [ 30%] Meshing curve 67 (Line)
Info    : [ 30%] Meshing curve 68 (Line)
Info    : [ 30%] Meshing curve 69 (Line)


Let's look at the mesh attributes -- we will use these to help us build our pypalace.Config object

In [6]:
attributes_table

,Name,ID,Type
0,cross,1,Surface
1,claw_connector_arm,2,Surface
2,josephson_junction,3,Surface
5,substrate,4,Volume
6,air,5,Volume
3,ground_plane,6,Surface
4,far_field,7,Surface


## Build and run our Palace simulation  

In [7]:
from pypalace import Config, Simulation
from pypalace.builder import Model, Domains, Boundaries, Solver

Define the necessary paths/filenames

In [8]:
''' path to Palace executable ''' 
path_to_palace = "/Users/firasabouzahr/Desktop/AWSPalace/install/bin/palace-arm64.bin" # change to your path

''' choose path & name for Palace configuration file we will build below '''
config_dir = "config/"
path_to_config = config_dir + "example00.json"

#### create Config object, define ["Problem"] block, & define ["Model"] block

In [9]:
my_config = Config(path_to_config)
 
# define config["Problem"]
my_config.add_Problem("Eigenmode",Output="example00_output")

# define config["Model"] block
my_config.add_Model(path_to_mesh,L0=1e-6) 

#### Add materials and boundary conditions (["Domains"] & ["Boundaries"] blocks, respectively)

In [10]:
# define materials
silicon = Domains.Material(Attributes = [4],Permeability=1.0,Permittivity=11.45,LossTan=1e-07)
air = Domains.Material([5],1.0,1.0,0.0)
my_materials = [silicon,air] # material list for input into add_Domains()

# define boundary conditions
PECs = Boundaries.PEC([1,2,6,7]) # add perfect electric conductors --> all metal components from Quantum Metal design + ground_plane
JJ = Boundaries.LumpedPort(Index=1,Attributes=[3],Direction="+Y",L=14e-09) # add josephson_junction polygon as inductive lumped port w/ LJ = 14 nH
my_BCs = [PECs,JJ] # boundary condition list for input into add_Boundaries()

# add config["Domains"] and config["Boundaries"] using our material and BC lists above
my_config.add_Domains(my_materials)
my_config.add_Boundaries(my_BCs)

#### Add ["Solver"] 

**Note:** The simulation parameters below are very coarse intentionally such that this simulation can be ran locally. For higher fidelity simulation results, update parameters below to be stricter. See pyPalace [Example 02](https://github.com/FirasAbouzahr/pyPalace/tree/main/Examples/example_01_eigenmode_EPR/example01_script.py) for an exmaple of running on an HPC.

In [11]:
# eigenmode parameters
eigenmode_params = Solver.Eigenmode(Target = 1.0,
                                    Tol = 1.0e-4,
                                    N = 1,
                                    Save = 1)
## linear solver parameters
Linear_params = Solver.Linear(Type="Default",
                              KSPType = "Default",
                              Tol = 1e-4,
                              MaxIts = 25)

## add them to config["Solver"] and solver["Linear"]
my_config.add_Solver(Simulation=eigenmode_params,Order = 1,Linear=Linear_params)

# save it
my_config.save_config(check_validity=True) # checks validity of file and raises error if something is missing

## Create simulation object & run this simulation locally

In [12]:
''' simulate our Quantum Metal design'''

my_sim = Simulation(my_config,path_to_palace) # define simulation object

n=5 # number of MPI ranks
my_sim.run(n=n)


_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/


--> Warning!
Output folder is not empty; program will overwrite content! (output_test)
Git changeset ID: v0.13.0-509-g360de9ab
Running with 5 MPI processes
Device configuration: cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked

Added 28 elements in 1 iterations of local bisection for under-resolved interior boundaries
Added 9723 duplicate vertices for interior boundaries in the mesh
Added 24301 duplicate boundary elements for interior boundaries in the mesh
Added 28067 boundary elements for material interfaces to the mesh
Finished partitioning mesh into 5 subdomains

Characteristic length and time scales:
 L₀ = 1.507e-03 m, t₀ = 5.027e-03 ns

Mesh curvature order: 1
Mesh bounding box:
 (Xmin, Ymin, Zmin) = (-7.200e-04, -7.200e-04, -5.000e-04) m
 (Xmax, Ymax, Z

## Extract and analyze simulation results

#### Visualize electric field to ensure mode 1 (index=1) is indeed our qubit

In [16]:
%matplotlib inline
my_sim.plot_field(field="E",index=1)

#### Extract transmon qubit frequency and anharmonicity with EPR

For a more in-depth look at EPR analysis see: pyPalace [Example 02 analysis notebook](https://github.com/FirasAbouzahr/pyPalace/tree/main/Examples/example_01_eigenmode_EPR/example01_analysis_notebook.ipynb)

In [14]:
from pypalace.analysis import EPR

f_q = my_sim.get_frequency_eigenmode(mode=1) # qubit frequency
p_q = my_sim.get_portEPR(port_index=1,mode=1) # qubit energy participation ratio to JJ for EPR analysis

alpha = EPR.calculate_anharmonicity(p_q,f_q,LJ=14e-09)

In [15]:
print("Transmon Hamiltonian parameters:")
print("--------------------------------")
print(f"qubit frequency = {f_q:.2f} GHz")
print(f"qubit anharmonicity = {alpha/1e6:.2f} MHz")

Transmon Hamiltonian parameters:
--------------------------------
qubit frequency = 3.72 GHz
qubit anharmonicity = -146.18 MHz
